In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import os
import json
import pandas as pd

In [2]:
data_path = "./data.xml"; assert os.path.exists(data_path), "File does not exist: {}".format(data_path)
date_path = "./dates.json"; assert os.path.exists(date_path), "File does not exist: {}".format(date_path)

In [3]:
root = ET.parse(data_path).getroot()
dates = json.load(open(date_path, "r"))

sleep_data = [r for r in root.findall("Record")]
print("Total number of records: {}".format(len(sleep_data)))
print("Total days of data: {}".format(len(dates)))

Total number of records: 4140
Total days of data: 85


In [11]:
def feature_extraction(dates, data):
    raw_df = pd.DataFrame([r.attrib for r in data])
    
    raw_df["startDate"] = pd.to_datetime(raw_df["startDate"])
    raw_df["endDate"] = pd.to_datetime(raw_df["endDate"])
    raw_df["duration"] = (raw_df["endDate"] - raw_df["startDate"]).dt.total_seconds() / 60.0
    
    rows = []
    weekday_map = {"Mon": 0, "Tue": 1, "Wed": 2, "Thu" :3, "Fri": 4, "Sat": 5, "Sun": 6}
    for weekday, indices in dates:
        session_slice = raw_df.iloc[indices]
        core = session_slice[session_slice["value"] == "HKCategoryValueSleepAnalysisAsleepCore"]["duration"].sum()
        deep = session_slice[session_slice["value"] == "HKCategoryValueSleepAnalysisAsleepDeep"]["duration"].sum()
        rem = session_slice[session_slice["value"] == "HKCategoryValueSleepAnalysisAsleepREM"]["duration"].sum()
        awake = session_slice[session_slice["value"] == "HKCategoryValueSleepAnalysisAwake"]["duration"].sum()
        start_time = session_slice["startDate"].min().hour + session_slice["startDate"].min().minute / 60.0
        rows.append([start_time, core, deep, rem, awake, weekday_map[weekday]])
    df = pd.DataFrame(rows, columns=["start_time", "CORE", "DEEP", "REM", "AWAKE", "weekday"])
    return df

def train_kmeans(df, n_clusters=5):
    df = df.dropna().copy()
    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df)
    
    model = KMeans(n_clusters=n_clusters, random_state=42)
    df["cluster"] = model.fit_predict(X_scaled)
    return df, model, scaler

In [12]:
df = feature_extraction(dates, sleep_data)

In [13]:
final_df, model, scaler = train_kmeans(df)
print(final_df.groupby('cluster').mean())

         start_time        CORE       DEEP         REM      AWAKE   weekday
cluster                                                                    
0          5.246825  294.263492  97.411111  119.106349   6.461905  1.476190
1          5.010215  126.226882  47.110215   52.829032   4.990860  5.064516
2         10.998718   96.880769  38.830769   26.782051   5.252564  1.461538
3          6.959259  152.374074  29.966667   59.662963  25.127778  2.888889
4          5.859259  343.955556  58.150000  207.788889  15.955556  4.111111
